In [ ]:
import numpy as np
import scipy.stats as stats

from dual_pfc_funcs import load_dict, getParams

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

# parameters
params = getParams()
subjects = params['subjects']
plot_sym = params['markers']
color_map = params['color_map']

In [ ]:
CROSSVAL = False
data_path = 'preprocessed_data/'
if CROSSVAL:
    dat = load_dict(data_path + 'all_pupil_prediction_cv.pkl')
else:
    dat = load_dict(data_path + 'all_pupil_prediction.pkl')
fnames = dat.keys()

In [ ]:
# example session pupil prediction
ex_dat = dat['Sa191206']
y = ex_dat['pupil_zsc']
print('Total trials in example session: N={}'.format(y.shape[0]))

# predicitons
yhat_across = ex_dat['predictions']['across']
yhat_within_left = ex_dat['predictions']['within-left']
yhat_within_right = ex_dat['predictions']['within-right']

print('R2 values - left {:.5f}, right {:.5f}, across {:.5f}'.format(ex_dat['r2']['within-left'], ex_dat['r2']['within-right'], ex_dat['r2']['across']))

fig,ax = plt.subplots(1,1)

start,end = 300,340
idx = np.arange(0,end-start,1)
ax.plot(idx,y[start:end],'k',label='Actual pupil')
ax.plot(idx,yhat_across[start:end],label='across pred', color=color_map['across'])
ax.plot(idx,yhat_within_left[start:end],label='within left pred', color=color_map['within2'])
ax.plot(idx,yhat_within_right[start:end],label='within right pred', color=color_map['within1'])

ax.legend(loc='best')
ax.set_xlabel('trial number (example session)')
ax.set_xticks(np.arange(0,end-start+1,10))
ax.set_xlim(-0.5,end-start+1)
ax.set_ylabel('trial-by-trial pupil diameter\n(normalized)')
ax.set_yticks(ticks=np.arange(-1,5,1))

if SAVE_FIG:
    pdf = PdfPages('figs/pupil_pred_ex_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
col = 'k'
fig,ax = plt.subplots(1,2,tight_layout=True,figsize=(12,5))
for i,latent in enumerate(['across','within-right','within-left']):
    for j,sub in enumerate(subjects):
        # trial-by-trial pupil
        curr = np.array([dat[sess]['r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()])
        if i==0:
            ax[0].errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6,label=f'Monkey {sub[:2].title()}')
        else:
            ax[0].errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6)
        
        # null dist
        null = np.array([dat[sess]['null_r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()]).flatten()
        null_prc = np.percentile(null,[0,95])
        ax[0].plot(i*3+j*.7,null.mean(),color='k',alpha=0.4,linewidth=3)
        ax[0].plot([i*3+j*.7,i*3+j*.7],null_prc,'k-',alpha=0.4,linewidth=3)

        # evoked pupil
        curr = np.array([dat[sess]['evoked']['r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()])
        ax[1].errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6)
        null = np.array([dat[sess]['evoked']['null_r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()]).flatten()
        null_prc = np.percentile(null,[0,95])
        ax[1].plot(i*3+j*.7,null.mean(),color='k',alpha=0.4,linewidth=3)
        ax[1].plot([i*3+j*.7,i*3+j*.7],null_prc,'k-',alpha=0.4,linewidth=3)
        
# formatting
ax[0].set_xticks(ticks=[0.7,3.7,6.7])
ax[0].set_yticks(ticks=np.arange(0,0.4,0.1))
ax[0].set_xticklabels(['across-area','within-area \n (right)','within-area \n (left)'])
ax[0].legend()
ax[0].set_ylabel('trial-by-trial pupil \n diameter prediction ($r^2$)')

# formatting
ax[1].set_xticks(ticks=[0.7,3.7,6.7])
ax[1].set_xticklabels(['across-area','within-area \n (right)','within-area \n (left)'])
ax[1].set_ylabel('evoked pupil \n diameter prediction ($r^2$)')

colors = [color_map['across'],color_map['within1'],color_map['within2']]
for xtick, color in zip(ax[0].get_xticklabels(), colors):
    xtick.set_color(color)
for xtick, color in zip(ax[1].get_xticklabels(), colors):
    xtick.set_color(color)

if SAVE_FIG:
    pdf = PdfPages('figs/pupil_pred_all_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# Statistics:
print('Trial-by-trial statistics')
all_across, all_left, all_right = np.array([]),np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([dat[sess]['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([dat[sess]['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([dat[sess]['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])

    # pool
    all_across = np.concatenate((all_across, across))
    all_left = np.concatenate((all_left, within_left))
    all_right = np.concatenate((all_right, within_right))

    p = stats.ttest_rel(a=across, b=within_left,alternative='greater').pvalue
    print(f'{sub} across > within-left, p={p:.6f}')
    p = stats.ttest_rel(a=across, b=within_right,alternative='greater').pvalue
    print(f'{sub} across > within-right, p={p:.6f}')

# pooled:
print()
print('Average r2 values: across {:.3f}, within left {:.3f}, within right {:.3f}'.format(all_across.mean(), all_left.mean(), all_right.mean()))
print('Pooled paired t test')
p = stats.ttest_rel(a=all_across, b=all_left,alternative='greater').pvalue
print(f'across > within-left, p={p:.6f}')
p = stats.ttest_rel(a=all_across, b=all_right,alternative='greater').pvalue
print(f'across > within-right, p={p:.6f}')

In [ ]:
# Statistics for evoked: ['evoked']
print('Evoked statistics')
all_across, all_left, all_right = np.array([]),np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([dat[sess]['evoked']['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([dat[sess]['evoked']['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([dat[sess]['evoked']['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])

    # pool
    all_across = np.concatenate((all_across, across))
    all_left = np.concatenate((all_left, within_left))
    all_right = np.concatenate((all_right, within_right))

    p = stats.ttest_rel(a=across, b=within_left,alternative='greater').pvalue
    print(f'{sub} across > within-left, p={p:.6f}')
    p = stats.ttest_rel(a=across, b=within_right,alternative='greater').pvalue
    print(f'{sub} across > within-right, p={p:.6f}')

# pooled:
print()
print('Average r2 values: across {:.3f}, within left {:.3f}, within right {:.3f}'.format(all_across.mean(), all_left.mean(), all_right.mean()))
print('Pooled paired t test')
p = stats.ttest_rel(a=all_across, b=all_left,alternative='greater').pvalue
print(f'across > within-left, p={p:.6f}')
p = stats.ttest_rel(a=all_across, b=all_right,alternative='greater').pvalue
print(f'across > within-right, p={p:.6f}')

In [ ]:
# repeat analysis with 1D predictions:
if CROSSVAL:
    control_dat = load_dict(data_path + 'all_pupil_prediction_1d_cv.pkl')
else:
    control_dat = load_dict(data_path + 'all_pupil_prediction_1d.pkl')
fnames = control_dat.keys()

# Statistics:
print('Trial-by-trial control')
all_across, all_left, all_right = np.array([]),np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([control_dat[sess]['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([control_dat[sess]['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([control_dat[sess]['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])
    # pool
    all_across = np.concatenate((all_across, across))
    all_left = np.concatenate((all_left, within_left))
    all_right = np.concatenate((all_right, within_right))

# pooled:
print('Average r2 values: across {:.3f}, within left {:.3f}, within right {:.3f}'.format(all_across.mean(), all_left.mean(), all_right.mean()))
print('Pooled paired t test')
p = stats.ttest_rel(a=all_across, b=all_left,alternative='greater').pvalue
print(f'across > within-left, p={p:.6f}')
p = stats.ttest_rel(a=all_across, b=all_right,alternative='greater').pvalue
print(f'across > within-right, p={p:.6f}')

print()
print('Evoked control')

# Statistics for evoked: ['evoked']
all_across, all_left, all_right = np.array([]),np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([control_dat[sess]['evoked']['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([control_dat[sess]['evoked']['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([control_dat[sess]['evoked']['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])
    # pool
    all_across = np.concatenate((all_across, across))
    all_left = np.concatenate((all_left, within_left))
    all_right = np.concatenate((all_right, within_right))

# pooled:
print('Average r2 values: across {:.3f}, within left {:.3f}, within right {:.3f}'.format(all_across.mean(), all_left.mean(), all_right.mean()))
print('Pooled paired t test')
p = stats.ttest_rel(a=all_across, b=all_left,alternative='greater').pvalue
print(f'across > within-left, p={p:.6f}')
p = stats.ttest_rel(a=all_across, b=all_right,alternative='greater').pvalue
print(f'across > within-right, p={p:.6f}')